# 06 — Streamlit demo on Kaggle (with public Cloudflare tunnel)

Runs `app/streamlit_app.py` inside a Kaggle session and exposes it via a free Cloudflare tunnel —
no ngrok account, no auth token, no port-forwarding setup.

**Setup before running**:
1. **Settings → Accelerator → GPU P100** (or T4 ×2) — needed to load Qwen2.5-7B in 4-bit.
2. **Settings → Internet → On**.
3. **Add-ons → Secrets → `HF_TOKEN`** attached (Read scope is enough — we only pull the LoRA adapter).
4. *(Optional, biggest speed-up)* **Add Input → Models → search `qwen2.5` → 7B-Instruct → Add**.
   This mounts the base model at `/kaggle/input/...` so there's no 15 GB download at all. If you
   skip this step, cell 6 still pre-downloads with `hf_transfer` (~2-3 min) so the public URL
   doesn't hang on first visit.
5. **Run all cells.**

**Why the link feels slow without these steps**: when a viewer clicks "Trả lời" on a cold demo,
Streamlit's `get_pipeline()` fires its first `from_pretrained` and starts pulling Qwen2.5-7B
(~15 GB) over the cloudflared tunnel. The tunnel times out long before the download finishes,
so the page just hangs.

**This notebook avoids that** by:
- enabling `hf_transfer` (3-5× faster HF downloads),
- pre-loading the model in cell 6 so the files are on Kaggle's local disk before the tunnel opens,
- setting `LAWMATE_PRELOAD_CONFIG=D` so Streamlit eager-loads on the first browser session
  instead of waiting for a button click.

After the last cell, you'll get a public URL like `https://<random>.trycloudflare.com` —
share it with anyone (judges, classmates) for the live demo. The URL stays alive for as long as
the Kaggle session runs (max 12 h).

In [ ]:
# 1. Clone (or update) the repo at /kaggle/working/LawMate.
import os, subprocess

REPO_URL = "https://github.com/tamir39/rag-llm-vietnam-law-advisor.git"
REPO_DIR = "/kaggle/working/LawMate"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "develop", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "origin", "develop"])

os.chdir(REPO_DIR)
print(subprocess.check_output(["git", "log", "-1", "--oneline"]).decode().strip())

In [ ]:
# 2. Install the LLM/RAG stack + Streamlit + hf_transfer (Rust-based parallel HF
#    downloader, 3-5x faster than the default for the 15GB Qwen base model).
%pip install -q -U "peft>=0.12" "trl>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33" "sentence-transformers>=3.0" "faiss-cpu>=1.8" "streamlit>=1.36" "hf_transfer>=0.1.8"

In [ ]:
# 3. HF login — needed to pull Tamir39/qwen2_5-7b-vietnam-tax-lora (and Qwen base if private/gated).
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF login OK")

In [ ]:
# 4. Build the FAISS index if it isn't there yet (~30s on Kaggle CPU).
import os, subprocess

if not os.path.isfile("experiments/index/kb.faiss"):
    subprocess.check_call(["python", "scripts/build_index.py"])
else:
    print("FAISS index already present — skipping build")

In [ ]:
# 5. Download the cloudflared binary (one-time, ~30 MB).
import os, stat, subprocess

BIN = "/kaggle/working/cloudflared"
if not os.path.isfile(BIN):
    subprocess.check_call([
        "wget", "-q", "-O", BIN,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    ])
    os.chmod(BIN, os.stat(BIN).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
print(subprocess.check_output([BIN, "--version"]).decode().strip())

In [ ]:
# 6. Download model + adapter + embedder files to disk WITHOUT loading them
#    into the notebook's Python process.
#
#    Why: torch.cuda.empty_cache() doesn't release GPU memory back to the OS -
#    it stays reserved by this kernel's CUDA context for the lifetime of the
#    notebook. If we instantiated the model here, the Streamlit subprocess
#    would have to fight us for VRAM and could deadlock on first user click.
#
#    snapshot_download just copies files to ~/.cache/huggingface/hub/, no
#    PyTorch import, no CUDA context. ~2-3 min with hf_transfer.
#
#    SKIP this cell if you've attached Qwen2.5-7B-Instruct as a Kaggle Model
#    (right-side panel -> Add Input -> Models -> qwen2.5 -> 7b-instruct) AND
#    pointed the configs at /kaggle/input/...; in that case there's no download.
import os, time

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download

REPOS = [
    "Qwen/Qwen2.5-7B-Instruct",            # base LLM (~15 GB)
    "Tamir39/qwen2_5-7b-vietnam-tax-lora",  # fine-tuned adapter (~160 MB)
    "intfloat/multilingual-e5-base",        # RAG embedder (~1 GB)
]

t0 = time.time()
for repo in REPOS:
    print(f"\n[{repo}] downloading...")
    snapshot_download(repo, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.md"])

print(f"\nAll files cached on disk in {time.time()-t0:.0f}s")
print("Notebook process holds no GPU memory. Streamlit subprocess will load fresh.")

In [ ]:
# 7. Launch Streamlit + cloudflared tunnel. Public URL prints at the end.
#
# Streamlit's stdout is redirected to /kaggle/working/streamlit.log so the
# subprocess never blocks on a full pipe buffer (Linux pipes are ~64 KB; once
# full, every print() inside Streamlit hangs and the script wedges with the
# model loaded but not generating). Tail the log live in another cell with:
#   !tail -f /kaggle/working/streamlit.log
import os, re, subprocess, time

os.chdir("/kaggle/working/LawMate")
ST_LOG = "/kaggle/working/streamlit.log"
CF_LOG = "/kaggle/working/cloudflared.log"

st_env = os.environ.copy()
st_env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

streamlit = subprocess.Popen(
    [
        "streamlit", "run", "app/streamlit_app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.address", "0.0.0.0",
        "--browser.gatherUsageStats", "false",
    ],
    stdout=open(ST_LOG, "w"), stderr=subprocess.STDOUT,
    env=st_env,
)

print(f"[streamlit] starting (logs: {ST_LOG})...")
started = False
with open(ST_LOG) as f:
    for _ in range(240):  # 60s max waiting for the URL line
        line = f.readline()
        if not line:
            time.sleep(0.25)
            continue
        print(line.rstrip())
        if "You can now view your Streamlit app" in line or "Network URL" in line:
            started = True
            break
if not started:
    raise RuntimeError(f"Streamlit failed to come up - see {ST_LOG}")

# Open cloudflared tunnel
print(f"\n[cloudflared] opening tunnel (logs: {CF_LOG})...")
tunnel = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8501"],
    stdout=open(CF_LOG, "w"), stderr=subprocess.STDOUT,
)

public_url = None
with open(CF_LOG) as f:
    for _ in range(240):
        line = f.readline()
        if not line:
            time.sleep(0.25)
            continue
        print(line.rstrip())
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            public_url = m.group(0)
            break

if not public_url:
    raise RuntimeError(f"Could not parse cloudflared URL - see {CF_LOG}")

print("\n" + "=" * 72)
print(f"  DEMO URL:  {public_url}")
print("=" * 72)
print("\nFollow logs in a separate cell:")
print(f"  !tail -f {ST_LOG}    # Streamlit (model load, errors, requests)")
print(f"  !tail -f {CF_LOG}    # Cloudflared (tunnel events)")
print("\nFirst question: ~30-60s spinner + 30-50s generation on T4 x2.")
print("Subsequent questions: ~10-30s each.")
print("Stop the demo by interrupting this cell or stopping the kernel.")

## Stopping the demo

- **Interrupt the cell above** (◼ button) — this kills both Streamlit and the tunnel.
- Or **Stop Kernel** from the right sidebar to shut down the whole session.
- Closing the Kaggle browser tab does **not** stop the session — your tunnel keeps running
  (and counts against your 30 GPU-hr/week quota) until the kernel idles out (~20 min).

## Troubleshooting

- **Tunnel URL prints but page shows error** — wait ~10 s and reload; cloudflared takes a moment
  to register the route after printing the URL.
- **"Address already in use" on port 8501** — a previous Streamlit didn't shut down. Restart the
  kernel (right sidebar → Stop session → Start) and re-run from cell 1.
- **OOM when loading Qwen** — switch accelerator to T4 ×2 (30 GB combined VRAM).